# 01 · Data Understanding
Exploration of the CDC Nutrition, Physical Activity, and Obesity (BRFSS) dataset.
This first step is **crucial** beacuse one is able to understand the variables, make cleaning processes when need it and have a broad perspective of the problem to solve ☝️  

# Data Validation   
At this first step, I'm trying to really understand how is shaped my dataset. This step is really important because I know what I'm working with.

In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv("../data/raw/Data_BRFSS.csv")
print(df.shape)
print(df.columns)

(110880, 33)
Index(['YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc', 'Datasource',
       'Class', 'Topic', 'Question', 'Data_Value_Unit', 'Data_Value_Type',
       'Data_Value', 'Data_Value_Alt', 'Data_Value_Footnote_Symbol',
       'Data_Value_Footnote', 'Low_Confidence_Limit', 'High_Confidence_Limit ',
       'Sample_Size', 'Total', 'Age(years)', 'Education', 'Sex', 'Income',
       'Race/Ethnicity', 'GeoLocation', 'ClassID', 'TopicID', 'QuestionID',
       'DataValueTypeID', 'LocationID', 'StratificationCategory1',
       'Stratification1', 'StratificationCategoryId1', 'StratificationID1'],
      dtype='str')


C:\Users\Juan David Castillo\AppData\Local\Temp\ipykernel_11196\3244810769.py:5: DtypeWarning: Columns (8: Data_Value_Unit) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/Data_BRFSS.csv")


In [3]:
space=75

df.columns=(
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

duplicates=df.duplicated().sum()

space=75
print("="*space)
print("Columns fixed".center(75))
print("="*space)
print("\n")
print(df.columns)

print("\n")
print("="*space)
print("Duplicates in the DataSet--->",duplicates)
print("="*space)
print("\n")

nulls=df.isnull().sum()
nulls=nulls[nulls>0].sort_values(ascending=False)
print("="*space)
msg = f"From the {df.shape[1]} columns in the dataset {len(nulls)} have nulls"
print(msg.center(75))
print("="*space)
print(nulls)




                               Columns fixed                               


Index(['yearstart', 'yearend', 'locationabbr', 'locationdesc', 'datasource',
       'class', 'topic', 'question', 'data_value_unit', 'data_value_type',
       'data_value', 'data_value_alt', 'data_value_footnote_symbol',
       'data_value_footnote', 'low_confidence_limit', 'high_confidence_limit',
       'sample_size', 'total', 'age_years', 'education', 'sex', 'income',
       'race_ethnicity', 'geolocation', 'classid', 'topicid', 'questionid',
       'datavaluetypeid', 'locationid', 'stratificationcategory1',
       'stratification1', 'stratificationcategoryid1', 'stratificationid1'],
      dtype='str')


Duplicates in the DataSet---> 0


              From the 33 columns in the dataset 15 have nulls             
total                         106920
data_value_unit               106260
sex                           102960
data_value_footnote            97666
data_value_footnote_symbol     97666
education   

## Actual Situation nulls
Taking a look to the data set, it's evident that some nulls have an obvious reason 🤔

For example, columns like sex, education and income it's because in that row it's not being measured that category (it's evident in column named *StratificationCategory1* ).
It's completelly the opposite case when the column *Data_Value* is null. In that case, the data set has no information, being evident in column *Data_Value_Footnote* when says: Data not available because sample size is insufficient or data not reported. In this case, is wortly to delete null values for work only with **reliable information**.


In [4]:
print(f"Actual nulls in data_value column---> {df['data_value'].isnull().sum()}")


df_clean=df.dropna(subset=["data_value"])
df_clean

print(f"Nulls after clean data_value column---> {df_clean['data_value'].isnull().sum()}")


df_clean.to_csv(r"C:\Users\Juan David Castillo\OneDrive\Python\prove12.csv")





Actual nulls in data_value column---> 13214
Nulls after clean data_value column---> 0


# Outliers
It's important to work only with valuable data. In this way, we are going to avoid bias in the results and the conclusions are going to be more accurate.

It worth it to see what's the most accurate method to discover outliers in this dataset. Generally, one can do it by groups or detect them by groups. In this analysis, I'm going to do by both generally and by groups, and see what insights brings each.


## The entire dataset


In [40]:
def outliers_detection(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    mask = (data[col] > upper_limit) | (data[col] < lower_limit)
    return mask, lower_limit, upper_limit


numeric_cols = ["data_value"]
outlier_mask = pd.Series([False] * len(df), index=df.index)

for col in numeric_cols:
    mask, lower, upper = outliers_detection(df, col)
    outlier_mask = outlier_mask | mask

df_outlier=df[outlier_mask]
df_clean=df[~outlier_mask]

outliers_count=df_outlier.shape[0]
print(f"Outliers detected by this method--->{outliers_count}")
print("\n")
count_answer=df_outlier['question'].value_counts().sort_values(ascending=False)

print("Outliers detected by question".center(80))
print("\n")
print(count_answer)





Outliers detected by this method--->2496


                         Outliers detected by question                          


question
Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)                                                                                    2236
Percent of adults aged 18 years and older who have obesity                                                                                                                                                                                                                                73
Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity (or an equivalent combination) and engage in muscle-strengthening activities on 2 or more days a week      40
Percent of

# Red flag 🚩
Detecting the outliers taking into account all the data set ( with out a group division) it's bringing **2496** outliers. However, as is evident below, almost 90% of those outliers come from the question *Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)*. Thus, is evident that this method does not really work. We have to divided into groups for really see what is an atipical value, and the most accurate method is grouping by **question**.

# Outliers detection by question ❓❔

In [43]:
def outliers_detection(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    mask = (data[col] > upper_limit) | (data[col] < lower_limit)
    return mask, lower_limit, upper_limit


col = "data_value"
outlier_mask = pd.Series([False] * len(df), index=df.index)
bounds = {}

for question, group in df.groupby("questionid"):
    mask, lower, upper = outliers_detection(group, col)
    outlier_mask.loc[group.index] = mask
    bounds[question] = (lower, upper)
    print(f"{question}: [{lower:6.2f}, {upper:6.2f}]  ->  {mask.sum():4d} outliers")

df_outlier = df[outlier_mask]
df_clean   = df[~outlier_mask]
print("\ntotal outliers:", outlier_mask.sum(), "| clean:", len(df_clean))

Q018: [ 23.25,  56.45]  ->    52 outliers
Q019: [  4.45,  36.85]  ->   148 outliers
Q036: [ 14.00,  49.20]  ->   634 outliers
Q037: [ 23.30,  45.70]  ->   527 outliers
Q043: [ 28.65,  73.85]  ->    66 outliers
Q044: [  4.25,  39.05]  ->    85 outliers
Q045: [ 13.50,  51.10]  ->   118 outliers
Q046: [  9.65,  54.85]  ->    67 outliers
Q047: [  3.40,  47.40]  ->   247 outliers

total outliers: 1944 | clean: 108936
